# Vision CISA · Evaluación de la selección con el modelo v3 (M3)

Corrida 2 de `docs/EVALUACION_SELECCION.md`. Compara, sobre los **mismos recortes**:

| Corrida | Qué usa | Dónde |
|---|---|---|
| 1b | calidad + pasada + hash + lectura de producción | aquí (CPU, ~1 min) |
| **2** | todo lo anterior **+ el modelo v3** (incertidumbre, desacuerdo, `sistema_distinto`, padrón) | aquí (GPU) |

**Antes:** `Entorno de ejecución → Cambiar tipo → GPU`. En Drive, carpeta `CISA_Reentrenamiento/` con:
`M3/delantero.zip` (recortes), `modelos/modelo_digitos_v3.pt` y `padron.csv`.

Resultados → `CISA_Reentrenamiento/M3/evaluacion_seleccion/`.

In [ ]:
# 0) Configuración
BASE      = "/content/drive/MyDrive/CISA_Reentrenamiento"
ZIP       = f"{BASE}/M3/delantero.zip"
PESOS_V3  = f"{BASE}/modelos/modelo_digitos_v3.pt"
PADRON    = f"{BASE}/padron.csv"
SALIDA    = f"{BASE}/M3/evaluacion_seleccion"
RAMA      = "mejora/reentrenamiento-eficiente"

In [ ]:
# 1) GPU, Drive, código y dependencias
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/vision-cisa && git clone -q -b {RAMA} https://github.com/visioncisa/vision-cisa.git /content/vision-cisa
%cd /content/vision-cisa
!git log --oneline -1
!pip install -q -r requirements-reentrenamiento.txt ultralytics==8.4.171
import os
for r in (ZIP, PESOS_V3, PADRON):
    print(("OK   " if os.path.exists(r) else "FALTA"), r)

In [ ]:
# 2) Recortes al disco local de Colab (mucho más rápido que leer de Drive)
!rm -rf /content/crudas && mkdir -p /content/crudas && unzip -q "{ZIP}" -d /content/crudas
!find /content/crudas -name "*.jpg" | wc -l

In [ ]:
# 3) Corrida 1b (sin modelo) y corrida 2 (con modelo v3)
!python -m reentrenamiento seleccionar --entrada /content/crudas --modelo m3 --salida /content/lote_1b
!python -m reentrenamiento preanotar --entrada /content/crudas --modelo m3 --pesos "{PESOS_V3}" --padron "{PADRON}" --salida /content/lote_2

In [ ]:
# 4) Comparación
import json, pandas as pd
m1 = json.load(open("/content/lote_1b/manifiesto.json"))["resumen"]
m2 = json.load(open("/content/lote_2/manifiesto.json"))["resumen"]
tabla = pd.DataFrame({"1b sin modelo": m1, "2 con v3": m2}).drop(index="razones")
display(tabla)
razones = pd.DataFrame({"1b sin modelo": m1["razones"], "2 con v3": m2["razones"]}).fillna(0).astype(int)
display(razones.sort_values("2 con v3", ascending=False))

TXT = {c: str for c in ("numero_sistema", "lectura_modelo", "consenso", "razones")}   # "0063" no debe volverse 63
r1 = pd.read_csv("/content/lote_1b/resumen.csv", dtype=TXT); r2 = pd.read_csv("/content/lote_2/resumen.csv", dtype=TXT)
e1 = set(r1[r1.decision != "descartar"].ruta); e2 = set(r2[r2.decision != "descartar"].ruta)
print(f"Elegidos en ambas: {len(e1 & e2)} · solo 1b: {len(e1 - e2)} · solo con modelo: {len(e2 - e1)}")

In [ ]:
# 5) ¿Qué tanto coincide el modelo v3 con lo que registró producción? (todos los recortes legibles)
r2["numero_sistema"] = r2.numero_sistema.fillna("").astype(str)
r2["lectura_modelo"] = r2.lectura_modelo.fillna("").astype(str)
legibles = r2[~r2.razones.fillna("").str.contains("mala_calidad")]
con_ambas = legibles[(legibles.numero_sistema != "") & (legibles.lectura_modelo != "")]
coinciden = (con_ambas.numero_sistema == con_ambas.lectura_modelo).mean()
print(f"Recortes con lectura de producción y del modelo: {len(con_ambas)} · coinciden: {coinciden:.1%}")
print("Diferencias más comunes (producción → modelo):")
display(con_ambas[con_ambas.numero_sistema != con_ambas.lectura_modelo]
        .groupby(["numero_sistema", "lectura_modelo"]).size().sort_values(ascending=False).head(20))

In [ ]:
# 6) Muestra estratificada para la auditoría humana (docs/EVALUACION_SELECCION.md §3)
import numpy as np
r2["grupo"] = np.where(r2.decision != "descartar", r2.decision, r2.razones.fillna("").str.split("|").str[-1])
tam = {"redundante": 100, "cupo_pasada": 100, "pasada_facil_fuera_de_muestra": 50, "revisar": 50, "conservar": 30}
partes = []
for g, n in tam.items():
    sub = r2[r2.grupo == g]
    if len(sub): partes.append(sub.sample(min(n, len(sub)), random_state=7))
mc = r2[r2.grupo.str.startswith("mala_calidad", na=False)]
if len(mc): partes.append(mc.sample(min(50, len(mc)), random_state=7))
auditoria = pd.concat(partes)[["grupo", "ruta", "pasada", "decision", "razones", "lectura_modelo", "numero_sistema"]]
auditoria["ruta"] = auditoria.ruta.str.replace("/content/crudas/", "", regex=False)
auditoria["util"] = ""      # ← quien revise escribe: si / no
print(auditoria.grupo.value_counts())

In [ ]:
# 7) Guardar todo en Drive
import shutil, os
os.makedirs(SALIDA, exist_ok=True)
for n in ("lote_1b", "lote_2"):
    for f in ("manifiesto.json", "resumen.csv"):
        shutil.copy(f"/content/{n}/{f}", f"{SALIDA}/{n}_{f}")
tabla.to_csv(f"{SALIDA}/comparacion.csv"); razones.to_csv(f"{SALIDA}/razones.csv")
auditoria.to_csv(f"{SALIDA}/auditoria.csv", index=False)
shutil.make_archive(f"{SALIDA}/lote_2_para_roboflow", "zip", "/content/lote_2")
print("Guardado en", SALIDA); print(os.listdir(SALIDA))

**Siguiente:** revisar `auditoria.csv` (columna `util`) y calcular los falsos descartes con intervalo de Wilson (§4 del documento).
`lote_2_para_roboflow.zip` trae los recortes elegidos **con las cajas del modelo v3** para corregir en Roboflow.